# OMX-UMI on Google Colab

Training, dataset / sidecar building, oracle-mask rendering and offline analysis.
The MuJoCo + ROS 2 closed-loop evaluators do **not** run on Colab (see ENVIRONMENT.md §4).

Runtime: GPU. Put the package zip, `sim_umi/data/` and the shared init / checkpoints on Google Drive first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# the package
!unzip -q -o /content/drive/MyDrive/omx_umi_github_3c3d9aa_20260928_v2.zip -d /content
%cd /content/omx-umi-sim

In [ ]:
# Python 3.9 training env (micromamba + pinned pip requirements), ~10 min
!bash colab/setup_colab.sh

In [ ]:
# configs and runner scripts use the training host's absolute path
!mkdir -p /home/j-j15a206 && ln -sfn /content/omx-umi-sim /home/j-j15a206/universal_manipulation_interface

In [ ]:
# data and the shared initial state from Drive (adjust the source paths)
!mkdir -p sim_umi/data sim_umi/train_results/2026-09-27_definitive/shared_init
!cp -r /content/drive/MyDrive/omx_umi_data/* sim_umi/data/
!cp /content/drive/MyDrive/omx_umi_ckpt/init_state.pt sim_umi/train_results/2026-09-27_definitive/shared_init/
# verify against the manifest
!grep -v '^#' manifests/data_sha256.txt | awk '{print $1"  "$3}' | sha256sum -c --quiet && echo DATA OK

In [ ]:
# unit tests (CPU + CUDA RNG check)
!bash -lc "cd /content/omx-umi-sim && source colab/activate.sh && CUDA_VISIBLE_DEVICES=0 python sim_umi/scripts/aux/unit_tests.py --out /content/unit_tests.json"

In [ ]:
# training: G0 (definitive recipe). MAX_STEPS=20 for a smoke, 11400 for the full run.
MAX_STEPS = 20
!bash -lc "cd /content/omx-umi-sim && source colab/activate.sh && CUDA_VISIBLE_DEVICES=0 WANDB_MODE=offline HYDRA_FULL_ERROR=1 python train.py --config-name=train_diffusion_unet_timm_omx_workspace task.dataset.val_ratio=0.0 training.seed=42 training.device=cuda:0 training.num_epochs=100 training.checkpoint_every=10 training.sample_every=5 training.resume=True +training.mixed_precision=fp16 +training.lr_scheduler_total_steps=11400 +training.checkpoint_at_steps=[5986,8000,10000,11400] +training.init_state_path=/content/omx-umi-sim/sim_umi/train_results/2026-09-27_definitive/shared_init/init_state.pt task.dataset_path=/content/omx-umi-sim/sim_umi/data/omx_sim_train_v2.zarr.zip +training.max_opt_steps={MAX_STEPS} exp_name=colab_g0 hydra.run.dir=/content/runs/colab_g0"
# ablations: --config-name=train_diffusion_unet_timm_omx_timing_aux_workspace (A) or ..._oracle_tip_relation_film_workspace (B)

In [ ]:
# oracle renderer vs the frozen sidecar (EGL)
!bash -lc "cd /content/omx-umi-sim && source colab/activate.sh && python sim_umi/scripts/aux/oracle_render_regression.py --demos 2 --out /content/render_regression.json"